In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import property_management
import tenants

SOURCE_PATH = "/Volumes/property_catalog/property_management/landing_zone/tenants.csv"
TARGET_SCHEMA = "property_catalog.property_management"

RAW_TABLE = f"{TARGET_SCHEMA}.raw_tenants"
CLEAN_TABLE = f"{TARGET_SCHEMA}.clean_tenants"
REJECTED_TABLE = f"{TARGET_SCHEMA}.rejected_tenants"

In [0]:
tenants_df = property_management.extract_property_management_data(spark, SOURCE_PATH)

# Incrementally load raw tenant records
property_management.upsert_property_management_table(spark, tenants_df, RAW_TABLE, key_column="Id")

display(tenants_df)

In [0]:
# Read raw tenants Delta table
tenants_df = spark.table(RAW_TABLE)

transformed_tenants_df = tenants.transform_tenants(tenants_df)

In [0]:
validated_tenants_df = tenants.validate_tenants(transformed_tenants_df)

display(validated_tenants_df)

In [0]:
clean_tenants_df, rejected_tenants_df = tenants.separate_tenants(validated_tenants_df)

print("Clean tenants:")
display(clean_tenants_df)

print("Rejected tenants:")
display(rejected_tenants_df)


In [0]:
tenants_summary = property_management.summarise_property_management_etl(tenants_df, clean_tenants_df, rejected_tenants_df, SOURCE_PATH)

In [0]:

# Remove validation flags from the clean output
clean_tenants_output_df = clean_tenants_df.drop(
    "invalid_tenant_id",
    "duplicate_tenant_id",
    "invalid_first_name",
    "invalid_last_name",
    "invalid_email",
    "invalid_lease_start_date",
    "invalid_lease_end_date",
    "invalid_lease_period",
    "invalid_property_id"
)

# Incrementally load clean tenant records
property_management.upsert_property_management_table(spark,clean_tenants_output_df,CLEAN_TABLE,key_column="Id")

# Incrementally load rejected tenant records
property_management.upsert_property_management_table(spark, rejected_tenants_df, REJECTED_TABLE, key_column="Id")

print(f"Clean tenants records saved to: {CLEAN_TABLE}")
print(f"Rejected tenants records saved to: {REJECTED_TABLE}")